In [10]:
import pandas as pd
import numpy as np
import xgboost
from sklearn.model_selection import RandomizedSearchCV
import pickle

In [11]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [12]:
train = train.drop(train[(train['GrLivArea'] > 4000)].index)

In [13]:
to_drop = ['Id', 'Alley', 'PoolQC', 'Fence', 'MiscFeature']
train.drop(columns=to_drop, inplace=True, errors='ignore')
test.drop(columns=to_drop, inplace=True, errors='ignore')

In [14]:
cat_col_train = ['FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual', 
                 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
                 'GarageQual', 'GarageCond']
ncat_col_train = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']

cat_col_test = ['FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual', 
                'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
                'GarageQual', 'GarageCond', 'MSZoning', 'Utilities', 'Exterior1st', 
                'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType']
ncat_col_test = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 
                 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 
                 'GarageCars', 'GarageArea']

In [15]:
for i in cat_col_train:
    train[i] = train[i].fillna(train[i].mode()[0])
for j in ncat_col_train:
    train[j] = train[j].fillna(train[j].mean())

for i in cat_col_test:
    test[i] = test[i].fillna(test[i].mode()[0])
for j in ncat_col_test:
    test[j] = test[j].fillna(test[j].mean())


In [18]:
final_df = pd.concat([train, test], axis=0)

final_df['TotalSF'] = final_df['TotalBsmtSF'] + final_df['1stFlrSF'] + final_df['2ndFlrSF']
final_df['HouseAge'] = final_df['YrSold'] - final_df['YearBuilt']

cols_to_drop = ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'YearBuilt']
final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns], inplace=True)

all_cat_col = ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 
               'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 
               'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 
               'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 
               'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 
               'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 
               'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
               'PavedDrive', 'SaleType', 'SaleCondition']

for field in all_cat_col:
    if field in final_df.columns:
        df_dummy = pd.get_dummies(final_df[field], drop_first=True, prefix=field)
        final_df.drop([field], axis=1, inplace=True)
        final_df = pd.concat([final_df, df_dummy], axis=1)

final_df = final_df.loc[:, ~final_df.columns.duplicated()]
num_train = len(train)
df_train = final_df.iloc[:num_train, :]
df_test = final_df.iloc[num_train:, :].drop(['SalePrice'], axis=1)

X_train = df_train.drop(['SalePrice'], axis=1)

y_train = np.log1p(df_train['SalePrice'])